# Stage 17 — GO / NO-GO gate: fingertip trajectory -> image
**Before any GPU.** Reuses the Stage 11 landmark cache (no MediaPipe re-run): the
index fingertip is landmark 8 -> `feature[:, 24:26]`. We render each clip's path
(hue = temporal order, width = inverse speed, flipped L-to-R) and show ~30 next to
their GT words. **If a human can't read a fair fraction, the VLM won't either —
fix tracking/rendering before training.**

Attach EITHER your **Stage 11 landmark-cache** dataset (instant) OR the **raw
122-signer frames** dataset (triggers a ~2-3 h MediaPipe extraction). Internet ON.


## Cell 1 — clone repo


In [ ]:
import sys, subprocess as sp
sp.run('rm -rf /kaggle/working/wita_v2', shell=True)
sp.run("git clone -b stage13b-paper-replication "
       "'https://github.com/Gaurs86/WiTA-v2.git' '/kaggle/working/wita_v2'",
       shell=True, check=True)
sys.path.insert(0, '/kaggle/working/wita_v2')
print('cloned')


## Cell 2 — locate the landmark cache


In [ ]:
import os
from stage17.common import find_landmark_cache
LM_CACHE = find_landmark_cache()
HAVE_CACHE = os.path.isdir(os.path.join(LM_CACHE, 'train'))
print('landmark cache:', LM_CACHE, '| present:', HAVE_CACHE)


## Cell 3 — (only if cache missing) extract landmarks  ~2-3 h
Skipped automatically if the Stage 11 cache is attached. Re-extraction is the
slow MediaPipe pass over ~10k clips; the faster path is to attach the cache.


In [ ]:
if not HAVE_CACHE:
    sp.run('pip -q install mediapipe', shell=True)
    from stage16.common import find_data_root
    from datasets.landmark_cache_122 import extract_dir_per_clip_landmarks_parallel
    RAW = find_data_root()
    LM_CACHE = '/kaggle/working/landmark_cache_122'
    for split in ['train', 'val', 'test']:
        for sub in ['lex', 'nonlex']:
            d = os.path.join(RAW, f'eng_{split}_{sub}')
            if os.path.isdir(d):
                extract_dir_per_clip_landmarks_parallel(d, LM_CACHE, split, sub, n_workers=4)
    HAVE_CACHE = os.path.isdir(os.path.join(LM_CACHE, 'train'))
print('using landmark cache at', LM_CACHE, '| present:', HAVE_CACHE)


## Cell 4 — detection-rate report (data quality)


In [ ]:
from stage17.gate import detection_report
_ = detection_report(LM_CACHE, split='train')
_ = detection_report(LM_CACHE, split='val')


## Cell 5 — random sample sheet (the GO/NO-GO read)


In [ ]:
from stage17.gate import sample_sheet
from IPython.display import Image as IPyImage, display
p = sample_sheet(LM_CACHE, '/kaggle/working/gate_random.png',
                 split='train', n=30, cols=6, flip_x=True)
display(IPyImage(filename=p))


## Cell 6 — worst-detection clips (stress test)


In [ ]:
p = sample_sheet(LM_CACHE, '/kaggle/working/gate_worst.png',
                 split='train', n=18, cols=6, worst_det=True, flip_x=True)
display(IPyImage(filename=p))


## Cell 7 — verify the horizontal flip on known words
Whichever side reads left-to-right as the GT word is the correct orientation.


In [ ]:
pf = sample_sheet(LM_CACHE, '/kaggle/working/gate_flip.png',
                  split='train', n=12, cols=6, flip_x=True,  seed=7)
pn = sample_sheet(LM_CACHE, '/kaggle/working/gate_noflip.png',
                  split='train', n=12, cols=6, flip_x=False, seed=7)
print('FLIP (expect L-to-R readable):'); display(IPyImage(filename=pf))
print('NO FLIP:');                       display(IPyImage(filename=pn))


## GO / NO-GO decision
- **GO** if you can read a fair fraction of the random sheet as their GT words
  → proceed to VLM fine-tuning (next notebook).
- **NO-GO** if most are unreadable scribbles → fix first: try `smooth_k`,
  `w_min/w_max`, color on/off, and especially **re-extract at higher `T_native`**
  (32 frames may be too coarse for long words). The worst-detection sheet shows
  whether tracking (not rendering) is the culprit.
